# Tạo và nạp chunk vào FAISS HNSW — Kaggle / Colab

Notebook độc lập: upload notebook, gắn/upload dữ liệu, sửa **`data_path`** ở cell cấu hình rồi **Run All**. Không cần index đã train hay module từ repository. Output mặc định tự chọn `/kaggle/working/faiss_hnsw`, `/content/faiss_hnsw` hoặc `./faiss_hnsw`.

Input là JSON array tài liệu giống `FAISS_index_chunks.ipynb`: `doc_id`, `status`, `url`, `children`. JSON gọn hoặc xuống dòng đều được đọc theo luồng. Output gồm `index.faiss` (HNSW, inner product trên vector float32 chuẩn hóa L2) và `metadata.json`, với ID vector liên tiếp từ 0 khớp key metadata.

**Cách chạy:**
1. Kaggle: bật Internet/GPU, Add Input dataset rồi sửa `data_path` (ví dụ `/kaggle/input/my-dataset/merged.json`). Colab: chọn GPU runtime, upload/mount dữ liệu rồi sửa `data_path` (ví dụ `/content/merged.json`).
2. Cell dependency chỉ kiểm tra và báo đúng lệnh cài khi thiếu; tự quyết định cài gói được báo rồi restart nếu đã import thư viện. Không tự cài hay thay PyTorch/CUDA/driver.
3. Có thể giới hạn `start_point` / `end_point` theo **doc_id, bao gồm cả hai đầu**. `end_point=-1` chạy đến hết. Đây không phải số dòng hay vị trí chunk; không giả định file sắp xếp theo doc_id.
4. Chạy các cell từ trên xuống. Khi chạy lại, chọn `output_dir` mới để bảo vệ output cũ. Nếu runtime đã báo `device-side assert triggered` / `illegal memory access`, **restart session/runtime** trước khi chạy lại; `empty_cache()` không sửa CUDA context hỏng.

HNSW tạo mới trên CPU, embedding ưu tiên GPU. Thiết kế này dùng được với `faiss-cpu` và tránh cố chuyển HNSW sang GPU vốn không được hỗ trợ trong quy trình FAISS này ([FAISS](https://github.com/facebookresearch/faiss/wiki/Guidelines-to-choose-an-index)). Model GTE cần Internet lần tải đầu; có thể dùng cache/local model qua `embedding_model`.

HNSW giữ vector đầy đủ và graph trong RAM: với d=768, M=32, ước lượng cơ bản khoảng **3.1 GiB / triệu vector**; 19 triệu vector khoảng **59 GiB**, chưa gồm model, batch, graph tầng trên và dung lượng cấp phát tạm. Streaming không giảm RAM của index. Dùng khoảng doc_id nhỏ phù hợp runtime nếu dữ liệu vượt RAM.

File đang ghi dùng `.partial`; không ghi đè input/output sẵn có. Notebook chưa hỗ trợ resume hay ghép nhiều lần chạy. Khi thất bại, giữ `.partial` để kiểm tra và dùng thư mục mới. Colab/Kaggle có output tạm: tải về hoặc lưu Drive trước khi runtime kết thúc.

## 1. Kiểm tra dependency


In [ ]:
from time import perf_counter

notebook_started = perf_counter()

import importlib.util
import sys

required_packages = {
    "numpy": "numpy",
    "torch": "torch",
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "transformers": "transformers",
}
missing_packages = [
    package for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing notebook dependencies: " + ", ".join(missing_packages)
        + ". No packages were installed. If approved, run: %pip install "
        + " ".join(missing_packages)
    )
print(f"Python: {sys.version.split()[0]}; dependency discovery passed")

In [ ]:
import gc
import json
import os
from importlib.metadata import version
from pathlib import Path
from time import perf_counter
from types import MethodType
from typing import Iterator, TextIO

# Set Hub timeouts before libraries initialize the download client.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

debug_cuda = False  # Enable only after restarting a failed CUDA runtime.
if debug_cuda:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

if Path("/kaggle/working").is_dir():
    runtime_dir = Path("/kaggle/working")
    input_dir = Path("/kaggle/input/YOUR_DATASET")
elif Path("/content").is_dir():
    runtime_dir = Path("/content")
    input_dir = runtime_dir
else:
    runtime_dir = Path.cwd()
    input_dir = runtime_dir

# Required edit: set the uploaded dataset path here.
data_path = input_dir / "merged.json"
output_dir = runtime_dir / "faiss_hnsw"

start_point = 0
end_point = -1  # Inclusive doc_id range; -1 means no upper bound.
embedding_model = "Alibaba-NLP/gte-multilingual-base"
model_code_revision = "40ced75c3017eb27626c9d4ea981bde21a2662f4"
dimension = 768
batch_size = 32  # Forward batch; automatically reduced after CUDA OOM.
add_batch_size = 1_024  # Buffer records for length sorting and batched HNSW add.
use_fp16 = True  # CUDA autocast only; stored vectors remain normalized float32.
require_gpu = False  # True forbids CPU fallback when CUDA is unavailable.
hnsw_m = 32
hnsw_ef_construction = 80  # Increase for recall, decrease for faster construction.
hnsw_ef_search = 64  # Search-time recall/latency; can be changed after loading.
faiss_threads = min(8, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1))
verify_saved_index = False  # True reloads the entire index and needs extra RAM.

for name in ("dimension", "batch_size", "add_batch_size", "hnsw_m",
             "hnsw_ef_construction", "hnsw_ef_search", "faiss_threads"):
    value = globals()[name]
    if type(value) is not int or value < 1:
        raise ValueError(f"{name} must be a positive integer")
if hnsw_m < 2 or hnsw_ef_construction < hnsw_m:
    raise ValueError("Use hnsw_m >= 2 and hnsw_ef_construction >= hnsw_m")
if (type(start_point) is not int or type(end_point) is not int
        or start_point < 0 or (end_point != -1 and end_point < start_point)):
    raise ValueError("Use integer start_point >= 0 and end_point = -1 or >= start_point")
if not data_path.is_file():
    raise FileNotFoundError(f"Input file not found; edit data_path: {data_path}")

output_index = output_dir / "index.faiss"
output_metadata = output_dir / "metadata.json"
partial_index = output_dir / "index.faiss.partial"
partial_metadata = output_dir / "metadata.json.partial"
for output_path in (output_index, output_metadata, partial_index, partial_metadata):
    if output_path.resolve() == data_path.resolve():
        raise ValueError("Output paths must differ from data_path")
    if output_path.exists():
        raise FileExistsError(f"Choose a new output_dir to preserve existing files: {output_path}")
output_dir.mkdir(parents=True, exist_ok=True)
faiss.omp_set_num_threads(faiss_threads)
torch.set_num_threads(faiss_threads)
print(f"torch={torch.__version__}; CUDA runtime={torch.version.cuda}; faiss={faiss.__version__}")
print(f"transformers={version('transformers')}; sentence-transformers={version('sentence-transformers')}")
print(f"doc_id range: [{start_point}, {end_point if end_point != -1 else 'unbounded'}]")
print(f"FAISS: CPU, {faiss_threads} threads; output: {output_dir}")

setup_seconds = perf_counter() - notebook_started


## 2. Tạo HNSW và tải model

Giữ model, pooling và chuẩn hóa L2 giống notebook mẫu. HNSW không cần train; dùng `index.add()` với ID ngầm định liên tiếp, thay cho `add_with_ids()` của IVFPQ. `M=32`, `efConstruction=80` là cấu hình khởi đầu cân bằng tốc độ/recall; không đảm bảo recall cho mọi dataset. `efSearch=64` áp dụng lúc search ([tham số FAISS](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes)).

GTE tải trên CPU, tái tạo buffer `position_ids`/RoPE theo cấu hình gốc và bổ sung attention-mask helper khi Transformers thiếu API cũ. Kiểm tra token ID, mask, vị trí trên CPU trước khi forward. Chạy smoke test CPU rồi thiết bị thực tế trước khi ghi metadata. Không tự giảm giới hạn token của model.

GPU hết VRAM: giảm batch và thử lại đúng chunk, kể cả runtime báo OOM dưới dạng `RuntimeError`. Batch=1 vẫn OOM thì dừng với hướng dẫn. FP16 cho embedding không hữu hạn thì chuyển FP32 cho các batch tiếp theo. CUDA assert/illegal access thì dừng và yêu cầu restart, không thử tiếp trên context hỏng. GPU không có: báo rõ chạy CPU hoặc dừng nếu `require_gpu=True`.


In [ ]:
model_started = perf_counter()

def prepare_gte_model(auto_model: torch.nn.Module) -> None:
    """Restore GTE's non-persistent buffers on CPU without changing learned weights."""
    if auto_model.config.model_type != "new":
        raise ValueError("This compatibility repair is specific to Alibaba GTE NewModel")
    embeddings = auto_model.embeddings
    expected_positions = torch.arange(
        auto_model.config.max_position_embeddings, dtype=torch.long, device="cpu",
    )
    embeddings.register_buffer("position_ids", expected_positions, persistent=False)
    # Reuse GTE's own initializer, including the original NTK scaling parameters.
    with torch.device("cpu"):
        embeddings._init_rope(auto_model.config)
    for name, buffer in embeddings.rotary_emb.named_buffers():
        if buffer.device.type != "cpu" or not torch.isfinite(buffer).all():
            raise ValueError(f"Invalid RoPE buffer after initialization: {name}")
    rotary = embeddings.rotary_emb
    if rotary.cos_cached.shape[0] < len(expected_positions):
        raise ValueError("RoPE cache is shorter than the configured position range")
    if not torch.equal(embeddings.position_ids, expected_positions):
        raise ValueError("GTE position_ids initialization failed")

    if not hasattr(auto_model, "get_extended_attention_mask"):
        def get_encoder_attention_mask(
            self, attention_mask: torch.Tensor, input_shape: tuple,
            device: torch.device | None = None, dtype: torch.dtype | None = None,
        ) -> torch.Tensor:
            """Reproduce the legacy bidirectional encoder padding mask."""
            if getattr(self.config, "is_decoder", False):
                raise ValueError("This compatibility helper supports encoder models only")
            if attention_mask.ndim == 2:
                extended_mask = attention_mask[:, None, None, :]
            elif attention_mask.ndim == 3:
                extended_mask = attention_mask[:, None, :, :]
            else:
                raise ValueError(f"Unsupported attention mask shape: {attention_mask.shape}")
            mask_dtype = self.dtype if dtype is None else dtype
            extended_mask = extended_mask.to(dtype=mask_dtype)
            return (1.0 - extended_mask) * torch.finfo(mask_dtype).min

        auto_model.get_extended_attention_mask = MethodType(
            get_encoder_attention_mask, auto_model,
        )


def validate_features(features: dict, auto_model: torch.nn.Module) -> None:
    """Reject invalid embedding indices on CPU before launching CUDA kernels."""
    input_ids = features["input_ids"]
    if input_ids.device.type != "cpu" or input_ids.dtype != torch.long:
        raise ValueError("Token IDs must be CPU int64 tensors before validation")
    if input_ids.ndim != 2 or not input_ids.numel():
        raise ValueError("Expected non-empty input_ids with shape (batch, sequence)")
    sequence_length = input_ids.shape[1]
    if sequence_length > auto_model.config.max_position_embeddings:
        raise ValueError("Token sequence exceeds the model's position limit")
    vocab_size = auto_model.get_input_embeddings().num_embeddings
    if input_ids.min().item() < 0 or input_ids.max().item() >= vocab_size:
        raise ValueError(f"Tokenizer IDs are outside the embedding vocabulary [0, {vocab_size})")
    mask = features.get("attention_mask")
    if mask is None or mask.shape != input_ids.shape or mask.device.type != "cpu":
        raise ValueError("attention_mask must match input_ids on CPU")
    if not torch.all((mask == 0) | (mask == 1)) or not torch.all(mask.sum(1) > 0):
        raise ValueError("Each attention_mask row must contain valid unmasked tokens")
    # Explicit positions also protect forward from a stale internal position buffer.
    features["position_ids"] = torch.arange(sequence_length).expand_as(input_ids)
    type_vocab_size = auto_model.config.type_vocab_size
    if type_vocab_size == 1:
        features["token_type_ids"] = torch.zeros_like(input_ids)
    elif type_vocab_size > 1 and "token_type_ids" in features:
        token_types = features["token_type_ids"]
        if (token_types.shape != input_ids.shape or token_types.dtype != torch.long
                or token_types.min().item() < 0 or token_types.max().item() >= type_vocab_size):
            raise ValueError("Invalid token_type_ids")


def embed_once(texts: list[str], mixed_precision: bool) -> np.ndarray:
    """Tokenize on CPU and run the existing SentenceTransformer pooling modules."""
    if callable(getattr(model, "preprocess", None)):
        features = model.preprocess(texts)
    else:
        features = model.tokenize(texts)
    validate_features(features, auto_model)
    features = {
        name: value.to(device) if isinstance(value, torch.Tensor) else value
        for name, value in features.items()
    }
    with torch.inference_mode(), torch.autocast(
        device_type=device.type, dtype=torch.float16,
        enabled=mixed_precision and device.type == "cuda",
    ):
        # This retains all configured pooling/normalization modules from the checkpoint.
        sentence_embeddings = model(features)["sentence_embedding"].float()
    vectors = sentence_embeddings.cpu().numpy()
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    if vectors.shape != (len(texts), index.d):
        raise ValueError(f"Unexpected embedding shape: {vectors.shape}; expected {(len(texts), index.d)}")
    if not np.isfinite(vectors).all():
        raise FloatingPointError("Model produced non-finite embeddings")
    norms = np.linalg.norm(vectors, axis=1)
    if np.any(norms <= 1e-12):
        raise FloatingPointError("Model produced zero embeddings")
    faiss.normalize_L2(vectors)
    return vectors


def encode_texts(texts: list[str]) -> np.ndarray:
    """Retry CUDA OOM with smaller batches, preserving every input and its order."""
    global effective_batch_size, use_fp16
    vectors = np.empty((len(texts), index.d), dtype=np.float32)
    # Group similar lengths to reduce padding; restore the original metadata order.
    order = sorted(range(len(texts)), key=lambda position: len(texts[position]), reverse=True)
    offset = 0
    mixed_precision = use_fp16 and device.type == "cuda"
    while offset < len(order):
        positions = order[offset:offset + effective_batch_size]
        retry_oom = False
        try:
            batch_vectors = embed_once([texts[position] for position in positions], mixed_precision)
        except torch.cuda.OutOfMemoryError as error:
            if device.type != "cuda":
                raise
            if len(positions) == 1:
                raise RuntimeError(
                    f"CUDA OOM for input {positions[0]} even at batch size 1. "
                    "Use a GPU with more VRAM; no chunk was skipped."
                ) from error
            effective_batch_size = max(1, len(positions) // 2)
            print(f"CUDA OOM; retrying with batch_size={effective_batch_size}")
            retry_oom = True
        except FloatingPointError:
            if not mixed_precision:
                raise
            mixed_precision = False
            use_fp16 = False
            print("Invalid FP16 embeddings; retrying this batch with FP32")
            continue
        except RuntimeError as error:
            message = str(error).lower()
            if "device-side assert" in message or "illegal memory access" in message:
                raise RuntimeError(
                    "CUDA context failed. Restart the Kaggle/Colab runtime and run from "
                    "the first cell; set debug_cuda=True for a synchronous traceback."
                ) from error
            if device.type == "cuda" and "out of memory" in message:
                if len(positions) == 1:
                    raise RuntimeError(
                        f"CUDA OOM for input {positions[0]} even at batch size 1. "
                        "Use a GPU with more VRAM; no chunk was skipped."
                    ) from error
                effective_batch_size = max(1, len(positions) // 2)
                print(f"CUDA OOM; retrying with batch_size={effective_batch_size}")
                retry_oom = True
            else:
                raise
        if retry_oom:
            # Retry only after the failed forward's exception frames have been released.
            gc.collect()
            torch.cuda.empty_cache()
            continue
        vectors[positions] = batch_vectors
        offset += len(positions)
    return vectors


def create_hnsw_index(embedding_dimension: int) -> faiss.IndexHNSWFlat:
    """Create an untrained CPU graph with cosine-compatible inner product scores."""
    new_index = faiss.IndexHNSWFlat(embedding_dimension, hnsw_m, faiss.METRIC_INNER_PRODUCT)
    new_index.hnsw.efConstruction = hnsw_ef_construction
    new_index.hnsw.efSearch = hnsw_ef_search
    return new_index


index = create_hnsw_index(dimension)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Embedding device: {device}")
if device.type == "cpu":
    if require_gpu:
        raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")
    print("CUDA is unavailable; embedding runs on CPU. Enable GPU in Kaggle/Colab for large data.")
else:
    try:
        torch.cuda.synchronize(device)
    except RuntimeError as error:
        raise RuntimeError("CUDA is not usable. Restart the runtime and enable GPU.") from error
    print(f"GPU: {torch.cuda.get_device_name(device)}")

model = SentenceTransformer(
    embedding_model, trust_remote_code=True, device="cpu",
    model_kwargs={"code_revision": model_code_revision},
    config_kwargs={"code_revision": model_code_revision},
)
model.float().eval()
auto_model = model[0].auto_model
prepare_gte_model(auto_model)
if model.get_sentence_embedding_dimension() != index.d:
    raise ValueError("Model embedding dimension differs from the configured HNSW dimension")
if model.max_seq_length > auto_model.config.max_position_embeddings:
    raise ValueError("Tokenizer sequence limit exceeds the GTE position limit")
if getattr(model, "default_prompt_name", None):
    raise ValueError("Expected the GTE checkpoint without a default embedding prompt")
effective_batch_size = batch_size
smoke_texts = ["Embedding smoke test.", "Kiểm tra embedding tiếng Việt với câu dài hơn."]
embedding_device = device
device = torch.device("cpu")
try:
    cpu_probe = encode_texts(smoke_texts)
finally:
    device = embedding_device
try:
    model.to(device)
except RuntimeError as error:
    raise RuntimeError(
        "Cannot move GTE to the selected device. If CUDA reports an assert/illegal "
        "access, restart the runtime. If out of memory, free VRAM or use a larger GPU."
    ) from error
probe = encode_texts(smoke_texts)
print(f"CPU and {device} embedding smoke tests passed: {probe.shape}")
print(f"HNSW: d={index.d}, M={hnsw_m}, efConstruction={index.hnsw.efConstruction}, efSearch={index.hnsw.efSearch}")
print(f"Approximate base index storage / million vectors: {(4 * index.d + 8 * hnsw_m) * 1_000_000 / 1024**3:.2f} GiB (additional overhead excluded)")

model_seconds = perf_counter() - model_started
print(f"Model setup and smoke tests: {model_seconds:.2f}s")


## 3. Embedding, thêm vector và ghi metadata theo batch

Lọc `status="ok"`, khoảng `doc_id`, child `retrieval_eligible` (mặc định True) và text không rỗng. Kiểm tra trường bắt buộc trước khi embedding, lỗi kèm doc/chunk. Metadata giữ cấu trúc notebook mẫu: `doc_id`, `chunk_order`, `char_start`, `char_end`, `chunk_text`, `chunk_id`, `parent_id`, `url`.

Tối ưu: đọc JSON theo luồng; gom `add_batch_size` record, sắp theo độ dài để giảm padding khi forward rồi phục hồi đúng thứ tự; `torch.inference_mode()`, autocast CUDA, add HNSW theo batch và CPU OpenMP đa luồng ([FAISS threading](https://github.com/facebookresearch/faiss/wiki/Threads-and-asynchronous-calls)). Không giữ toàn bộ text/embedding/metadata trong RAM. Graph đa luồng có thể khác giữa các lần build, ID vector vẫn khớp thứ tự metadata.

Log mỗi batch: tổng vector, chunk/giây trung bình (gồm đọc dữ liệu, embedding và add), thời gian embedding/add và batch GPU hiện tại. `add_batch_size` không phải batch GPU. Với runtime nhỏ, giảm `batch_size`/`add_batch_size`; giảm `efConstruction` tăng tốc build nhưng có thể giảm recall.

Bộ đọc giới hạn mỗi document ở 64 MiB ký tự để tránh tích lũy file JSON hỏng. Khi lỗi, không công bố output cuối; `.partial` có thể chưa là JSON hoàn chỉnh. Không hỗ trợ resume.


In [ ]:
def iter_merged_documents(
    data_file: TextIO, read_size: int = 64 * 1024,
    max_document_chars: int = 64 * 1024**2,
) -> Iterator[dict]:
    """Stream a JSON array in bounded blocks, independent of line formatting."""
    if read_size < 1 or max_document_chars < read_size:
        raise ValueError("Use 0 < read_size <= max_document_chars")
    decoder = json.JSONDecoder()
    buffer = ""
    position = 0
    eof = False
    document_number = 0

    def read_more() -> None:
        nonlocal buffer, position, eof
        buffer = buffer[position:]
        position = 0
        block = data_file.read(read_size)
        if block:
            buffer += block
        else:
            eof = True

    def skip_whitespace() -> None:
        nonlocal position
        while True:
            while position < len(buffer) and buffer[position] in " \t\r\n":
                position += 1
            if position < len(buffer) or eof:
                return
            read_more()

    read_more()
    if buffer.startswith("\ufeff"):
        position += 1
    skip_whitespace()
    if position >= len(buffer) or buffer[position] != "[":
        raise ValueError("Expected merged.json to contain a JSON array")
    position += 1
    skip_whitespace()
    if position < len(buffer) and buffer[position] == "]":
        position += 1
    else:
        while True:
            skip_whitespace()
            if position >= len(buffer) or buffer[position] != "{":
                raise ValueError(f"Expected a document object at document {document_number + 1}")
            while True:
                try:
                    document, end_position = decoder.raw_decode(buffer, position)
                except json.JSONDecodeError as error:
                    if eof:
                        raise ValueError(f"Invalid or incomplete JSON at document {document_number + 1}") from error
                    if len(buffer) - position >= max_document_chars:
                        raise ValueError(f"Document {document_number + 1} exceeds max_document_chars or has invalid JSON") from error
                    read_more()
                else:
                    if end_position - position > max_document_chars:
                        raise ValueError(f"Document {document_number + 1} exceeds max_document_chars")
                    position = end_position
                    break
            document_number += 1
            yield document
            skip_whitespace()
            if position >= len(buffer):
                raise ValueError("Missing closing ']' in merged.json")
            separator = buffer[position]
            position += 1
            if separator == "]":
                break
            if separator != ",":
                raise ValueError(f"Missing document separator after document {document_number}")
            skip_whitespace()
            if position < len(buffer) and buffer[position] == "]":
                raise ValueError("Trailing comma in merged.json")
    skip_whitespace()
    if position < len(buffer):
        raise ValueError("Unexpected content after merged.json array")


def iter_chunk_records(
    data_file: TextIO, start_doc_id: int = 0, end_doc_id: int = -1,
) -> Iterator[dict]:
    """Flatten eligible children and preserve the existing metadata field names."""
    if (type(start_doc_id) is not int or type(end_doc_id) is not int
            or start_doc_id < 0 or (end_doc_id != -1 and end_doc_id < start_doc_id)):
        raise ValueError("Invalid doc_id range")
    for document in iter_merged_documents(data_file):
        doc_id = document.get("doc_id")
        if type(doc_id) is not int:
            raise ValueError("Each document must have an integer doc_id")
        if doc_id < start_doc_id or (end_doc_id != -1 and doc_id > end_doc_id):
            continue
        if document.get("status") != "ok":
            continue
        if not isinstance(document.get("children"), list):
            raise ValueError(f"Expected a children list in document {doc_id}")
        if not isinstance(document.get("url"), str):
            raise ValueError(f"Expected a URL string in document {doc_id}")
        for child_number, chunk in enumerate(document["children"]):
            context = f"document {doc_id}, child {child_number}"
            if not isinstance(chunk, dict):
                raise ValueError(f"Expected a child object in {context}")
            if not chunk.get("retrieval_eligible", True):
                continue
            text = chunk.get("chunk_text")
            if not isinstance(text, str):
                raise ValueError(f"Non-string chunk_text in {context}")
            if not text.strip():
                continue
            required = ("doc_id", "chunk_order", "start_char", "end_char", "chunk_id", "parent_id")
            missing = [name for name in required if name not in chunk]
            if missing:
                raise ValueError(f"Missing fields {missing} in {context}")
            if type(chunk["doc_id"]) is not int or chunk["doc_id"] != doc_id:
                raise ValueError(f"Child doc_id differs from document in {context}")
            for name in ("chunk_order", "start_char", "end_char"):
                if type(chunk[name]) is not int or chunk[name] < 0:
                    raise ValueError(f"Expected a non-negative integer {name} in {context}")
            if chunk["end_char"] < chunk["start_char"]:
                raise ValueError(f"end_char precedes start_char in {context}")
            for name in ("chunk_id", "parent_id"):
                if not isinstance(chunk[name], str) or not chunk[name]:
                    raise ValueError(f"Expected a non-empty string {name} in {context}")
            yield {
                "doc_id": doc_id,
                "chunk_order": chunk["chunk_order"],
                "char_start": chunk["start_char"],
                "char_end": chunk["end_char"],
                "chunk_text": text,
                "chunk_id": chunk["chunk_id"],
                "parent_id": chunk["parent_id"],
                "url": document["url"],
            }


def add_batch(records: list[dict], metadata_file: TextIO, metadata_count: int) -> int:
    """Add sequential vector IDs and stream matching metadata without whole-file reads."""
    global embedding_total_seconds, add_total_seconds
    if not records:
        return metadata_count
    if index.ntotal != metadata_count:
        raise ValueError("FAISS vector count differs from metadata count before adding")
    embedding_started = perf_counter()
    vectors = encode_texts([record["chunk_text"] for record in records])
    embedding_seconds = perf_counter() - embedding_started
    ids = np.arange(metadata_count, metadata_count + len(records), dtype=np.int64)
    add_started = perf_counter()
    # IndexHNSWFlat assigns IDs in input order; no extra IDMap storage is needed.
    index.add(vectors)
    add_seconds = perf_counter() - add_started
    embedding_total_seconds += embedding_seconds
    add_total_seconds += add_seconds
    if index.ntotal != metadata_count + len(records):
        raise ValueError("FAISS did not add the expected number of vectors")
    for vector_id, record in zip(ids, records):
        if metadata_count:
            metadata_file.write(",\n")
        metadata_file.write(
            f'"{int(vector_id)}":' + json.dumps(record, ensure_ascii=False, separators=(",", ":"))
        )
        metadata_count += 1
    metadata_file.flush()
    elapsed = max(perf_counter() - build_started, 1e-9)
    print(
        f"Vectors: {index.ntotal:,}; elapsed={elapsed:.2f}s; {index.ntotal / elapsed:.1f} chunks/s; "
        f"embedding={embedding_seconds:.2f}s; add={add_seconds:.2f}s; "
        f"forward_batch={effective_batch_size}", flush=True,
    )
    return metadata_count


def build_chunk_index() -> int:
    """Build once per loaded empty index; keep incomplete metadata explicitly marked."""
    global build_complete, build_started, build_seconds
    global embedding_total_seconds, add_total_seconds
    build_complete = False
    build_started = perf_counter()
    build_seconds = 0.0
    embedding_total_seconds = 0.0
    add_total_seconds = 0.0
    if index.ntotal != 0:
        raise ValueError("Create an empty HNSW index before rerunning the build cell")
    for path in (output_index, output_metadata, partial_index, partial_metadata):
        if path.exists():
            raise FileExistsError(f"Choose a new output_dir before rerunning: {path}")
    count = 0
    batch = []
    with data_path.open(encoding="utf-8-sig") as data_file, partial_metadata.open("x", encoding="utf-8") as metadata_file:
        metadata_file.write("{\n")
        for record in iter_chunk_records(data_file, start_point, end_point):
            batch.append(record)
            if len(batch) >= add_batch_size:
                count = add_batch(batch, metadata_file, count)
                batch.clear()
        if batch:
            count = add_batch(batch, metadata_file, count)
        metadata_file.write("\n}\n")
    if count != index.ntotal:
        raise ValueError("FAISS vector count differs from metadata count")
    build_seconds = perf_counter() - build_started
    print(f"\nTotal vectors: {count:,}; build time: {build_seconds:.2f}s")
    return count


build_complete = False
metadata_count = build_chunk_index()
build_complete = True


## 4. Lưu output

Chỉ lưu khi build hoàn tất và số metadata khớp `index.ntotal`. Ghi `index.faiss.partial`, kiểm tra lượng byte tối thiểu chứa vector rồi đổi tên hai file sang `index.faiss` / `metadata.json`; không ghi đè output cũ.

HNSW không có cơ chế mmap tiết kiệm RAM như inverted lists của IVFPQ: mặc định **không đọc lại toàn bộ index** lúc lưu. `verify_saved_index=True` bật kiểm tra round-trip loại index, dimension, metric, count, tham số graph và vài vector; cần đủ RAM cho một bản index thứ hai. Lệnh `faiss.write_index()` và kiểm tra kích thước mặc định không xác nhận toàn bộ nội dung file bằng cách đọc lại.

Hai lần rename không phải giao dịch atomic của cả cặp file; nếu phiên bị ngắt giữa hai lần rename, không dùng output cho đến khi có đủ cả index và metadata. Không chạy nhiều notebook cùng ghi một `output_dir`.

Sau khi tải bằng `faiss.read_index()`, chuẩn hóa L2 query và đặt `index.hnsw.efSearch` để chỉnh search. ID search `i` ánh xạ đến `metadata[str(i)]`; `-1` là không có kết quả. HNSW dùng `efSearch`, không dùng `nprobe` của IVFPQ. Notebook không sửa loader backend. Metadata lớn cần cách tra cứu phù hợp ở ứng dụng để tránh tải cả JSON vào RAM.


Sau khi lưu thành công, notebook in **Runtime summary**: setup môi trường, tải model + smoke test, build, tổng embedding, tổng HNSW add, lưu + kiểm tra và **tổng thời gian `HH:MM:SS`**. Build đã bao gồm embedding/add nên không cộng lại các số này. Mỗi batch cũng hiển thị thời gian build đã chạy (`elapsed`).

Đo bằng `perf_counter()` từ cell dependency đến khi lưu xong; gồm tải model lần đầu. Khi chạy thủ công từng cell, tổng wall time gồm cả thời gian chờ giữa các cell; dùng **Run All** để đo một lượt liên tục. Thời gian từng giai đoạn model/build/save đo riêng lúc giai đoạn đó thực thi. Embedding đã đồng bộ GPU khi chuyển kết quả về NumPy trên CPU, nên không chỉ đo thời gian gửi kernel.


In [ ]:
def save_chunk_index() -> None:
    """Save the completed CPU graph while avoiding an unconditional second RAM copy."""
    global save_seconds, total_runtime_seconds
    if not build_complete or metadata_count != index.ntotal:
        raise ValueError("Run the build cell successfully before saving")
    if not isinstance(index, faiss.IndexHNSWFlat) or index.metric_type != faiss.METRIC_INNER_PRODUCT:
        raise ValueError("Expected a cosine-compatible IndexHNSWFlat")
    if not partial_metadata.is_file():
        raise FileNotFoundError("Completed partial metadata is missing")
    for path in (output_index, output_metadata, partial_index):
        if path.exists():
            raise FileExistsError(f"Output already exists: {path}")
    with partial_index.open("xb"):
        pass
    save_started = perf_counter()
    faiss.write_index(index, str(partial_index))
    if partial_index.stat().st_size < max(1, index.ntotal * index.d * 4):
        raise ValueError("Saved HNSW file is smaller than its raw vector storage")
    if verify_saved_index:
        saved_index = faiss.read_index(str(partial_index))
        if (not isinstance(saved_index, faiss.IndexHNSWFlat)
                or saved_index.ntotal != metadata_count or saved_index.d != index.d
                or saved_index.metric_type != index.metric_type
                or saved_index.hnsw.efConstruction != index.hnsw.efConstruction
                or saved_index.hnsw.efSearch != index.hnsw.efSearch
                or saved_index.hnsw.nb_neighbors(0) != index.hnsw.nb_neighbors(0)):
            raise ValueError("Saved HNSW index failed round-trip validation")
        if index.ntotal:
            sample_ids = sorted({0, index.ntotal // 2, index.ntotal - 1})
            for vector_id in sample_ids:
                if not np.array_equal(saved_index.reconstruct(vector_id), index.reconstruct(vector_id)):
                    raise ValueError(f"Saved vector differs at ID {vector_id}")
        del saved_index
    for partial_path, final_path in ((partial_index, output_index), (partial_metadata, output_metadata)):
        if final_path.exists():
            raise FileExistsError(f"Output already exists: {final_path}")
        partial_path.rename(final_path)
    save_seconds = perf_counter() - save_started
    print(f"Saved {index.ntotal:,} vectors in {save_seconds:.2f}s")
    print(f"Index: {output_index} ({output_index.stat().st_size / 1024**3:.3f} GiB)")
    print(f"Metadata: {output_metadata}")
    total_runtime_seconds = perf_counter() - notebook_started
    hours, remainder = divmod(int(total_runtime_seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    print("\nRuntime summary:")
    print(f"  Environment setup: {setup_seconds:.2f}s")
    print(f"  Model loading and smoke tests: {model_seconds:.2f}s")
    print(f"  Build (read, embed, add, metadata): {build_seconds:.2f}s")
    # Embedding includes tokenization, retries and the synchronizing CPU copy.
    print(f"    Embedding: {embedding_total_seconds:.2f}s")
    print(f"    HNSW add: {add_total_seconds:.2f}s")
    print(f"  Save and optional validation: {save_seconds:.2f}s")
    print(
        f"  Total notebook wall time: {total_runtime_seconds:.2f}s "
        f"({hours:02d}:{minutes:02d}:{seconds:02d})"
    )


save_chunk_index()
